# 04 - Treatment episodes, restricted to users without pre-pregnancy T2D

**Runs in:** Truveta Studio notebook environment only.

**What this notebook does**

Repeats the episode / persistence / exposure calculations of notebook 03 on the
subgroup with **no type 2 diabetes before pregnancy** (`t2d_before_pregnancy ==
False`), i.e. semaglutide users whose treatment is presumed weight-related.
This supports the sensitivity analysis reported alongside the main results.

The only substantive difference from notebook 03 is the cohort filter in
section 2; everything else is the same logic.

**Inputs** (from notebook 01):

* `medication_full.csv`
* `test_t3.csv`

**Output:**

| File | Contents |
|---|---|
| `revise_results/drugexposurenot2d.csv` | persistence / exposure table for this subgroup, grouped by `Discontinued60` |

**Run order:** 01 -> 02 -> 03 -> **04** -> 05.

## 1. Setup and configuration

In [ ]:
from truveta.study import Client, OutputMode, display_df

import re
import warnings

import numpy as np
import pandas as pd
import pyspark.pandas as ps

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================

POPULATION_TITLE = "Delivery"

GAP_DAYS = 60          # gap longer than this starts a new treatment episode

RESTRICT_TO_NO_PRIOR_T2D = True   # this is what distinguishes 04 from 03

IN_MEDICATION = "/medication_full.csv"
IN_COHORT     = "/test_t3.csv"

OUT_TABLE = "/revise_results/drugexposurenot2d.csv"

In [ ]:
client = Client(output_mode=OutputMode.PandasOnSpark)
# client = Client(output_mode=OutputMode.PySpark)

study = client.get_study()
population = study.get_population(title=POPULATION_TITLE)
snapshot = population.get_latest_snapshot()

output_path_local = study.get_output_path(fs=True)

## 2. Load inputs and restrict the cohort

In [ ]:
medication_full = pd.read_csv(output_path_local + IN_MEDICATION)
delivery_df = pd.read_csv(output_path_local + IN_COHORT)
print("exposed cohort:", delivery_df.shape)

if RESTRICT_TO_NO_PRIOR_T2D:
    delivery_df = delivery_df[delivery_df.t2d_before_pregnancy == False]
    print("restricted to no pre-pregnancy T2D:", delivery_df.shape)

In [ ]:
medication_df = medication_full[medication_full["PersonId"].isin(delivery_df["PersonId"])].copy()
medication_df = pd.merge(
    delivery_df[["PersonId", "delivery_date", "estimated_LMP"]],
    medication_df, on="PersonId", how="left")
print("dispensing records in subgroup:", medication_df.shape)

## 3. Build treatment episodes

Identical to notebook 03 section 3.

In [ ]:
def extract_formulation(code):
    """Pull the brand name out of a medication code description."""
    match = re.search(r"\[(.*?)\]", code)
    if match:
        return match.group(1).strip()
    for brand in ("Rybelsus", "Ozempic", "Wegovy"):
        if brand in code:
            return brand
    return "Unknown"


medication_df = medication_df.rename(columns={"DispenseDateTime": "med_date"})
medication_df["med_date"] = pd.to_datetime(medication_df["med_date"])
medication_df = medication_df[medication_df["med_date"] <= medication_df["delivery_date"]]
medication_df = medication_df.sort_values(["PersonId", "med_date"])
medication_df["Formulation"] = medication_df["Code"].apply(extract_formulation)

In [ ]:
medication_df = medication_df.groupby(["PersonId", "med_date"], as_index=False).agg({
    "DaysSupply": "sum",
    "Code": lambda x: ", ".join(sorted(set(x))),
    "delivery_date": "first",
    "estimated_LMP": "first",
    "Formulation": lambda x: ", ".join(sorted(set(x))),
})
medication_df["EndDate"] = medication_df["med_date"] + pd.to_timedelta(
    medication_df["DaysSupply"], unit="d")
medication_df.head()

In [ ]:
def assign_episodes(medication_df, gap_days=GAP_DAYS):
    """Chain fills into treatment episodes (new episode after a gap > `gap_days`)."""
    records = []

    for _, group in medication_df.sort_values(["PersonId", "med_date"]).groupby("PersonId"):
        group = group.reset_index(drop=True)
        episode_id = 1
        prev_end = group.loc[0, "EndDate"]

        group.loc[0, "EpisodeID"] = episode_id
        records.append(group.loc[0])

        for i in range(1, len(group)):
            gap = (group.loc[i, "med_date"] - prev_end).days
            if gap > gap_days:
                episode_id += 1
            group.loc[i, "EpisodeID"] = episode_id
            prev_end = group.loc[i, "EndDate"]
            records.append(group.loc[i])

    return pd.DataFrame(records)


episode_annotated_df = assign_episodes(medication_df)
episode_annotated_df.head()

In [ ]:
episode_df = episode_annotated_df.sort_values(["PersonId", "med_date"]).copy()
episode_df["NextStart"] = episode_df.groupby("PersonId")["med_date"].shift(-1)
episode_df["DiscontinuationTime"] = (episode_df["NextStart"] - episode_df["EndDate"]).dt.days
episode_df["IsReinitiation"] = (
    (episode_df["DiscontinuationTime"] > GAP_DAYS) & episode_df["NextStart"].notna())
episode_df.head()

## 4. Person-level treatment summary

Note this notebook does **not** carry `Supply` (median days supply); that column
is produced by notebook 03 and written to `drug_dayssupply.csv`.

In [ ]:
summary = episode_df.groupby("PersonId").agg(
    DrugStart=("med_date", "min"),
    DrugEnd=("EndDate", "max"),
    NumDrugEpisodes=("EpisodeID", lambda x: len(set(x))),
    Discontinued60=("DiscontinuationTime", lambda x: any(x > GAP_DAYS)),
    TotalDiscontinuationTime=("DiscontinuationTime", lambda x: x[x > GAP_DAYS].sum()),
    Reinitiation=("IsReinitiation", "any"),
).reset_index()

print("persons:", len(summary))
summary.head()

## 5. Accumulated persistence

In [ ]:
def accumulated_persistence(episode_df, anchor_col, out_col):
    """Sum episode durations, truncating each episode at `anchor_col`."""
    ep = episode_df.groupby(["PersonId", "EpisodeID"]).agg(
        EpisodeStart=("med_date", "min"),
        EpisodeEnd=("EndDate", "max"),
        anchor=(anchor_col, "first"),
    ).reset_index()

    ep["anchor"] = pd.to_datetime(ep["anchor"])
    ep["EpisodeEnd_Truncated"] = ep[["EpisodeEnd", "anchor"]].min(axis=1)

    # Rows with a missing anchor date fall out here (NaT comparisons are False).
    ep = ep[ep["EpisodeStart"] <= ep["anchor"]]

    ep["duration"] = (ep["EpisodeEnd_Truncated"] - ep["EpisodeStart"]).dt.days

    return ep, ep.groupby("PersonId").agg(**{out_col: ("duration", "sum")}).reset_index()


delivery_episode_summary, accumulated_delivery = accumulated_persistence(
    episode_df, "delivery_date", "AccumulatedPersistenceBeforeDelivery")

preg_episode_summary, accumulated_preg_persistence = accumulated_persistence(
    episode_df, "estimated_LMP", "AccumulatedPersistenceBeforePregnancy")

print("episodes before pregnancy:", len(preg_episode_summary),
      "| persons:", preg_episode_summary.PersonId.nunique())
accumulated_preg_persistence.head()

## 6. Exposure during pregnancy

In [ ]:
def pregnancy_exposure(row):
    """(was_exposed, exposure_days) for one fill's coverage window."""
    start, end, preg_start = row["med_date"], row["EndDate"], row["estimated_LMP"]

    if end <= preg_start:
        return (False, 0)

    exposure_days = (end - start).days if start >= preg_start else (end - preg_start).days
    return (True, exposure_days)


check_preg = episode_annotated_df.copy()
for c in ["estimated_LMP", "EndDate", "med_date"]:
    check_preg[c] = pd.to_datetime(check_preg[c])

check_preg[["WasExposedInPregnancy", "PregnancyExposureDays"]] = check_preg.apply(
    pregnancy_exposure, axis=1, result_type="expand")

pregnancy_df = check_preg.groupby("PersonId").agg(
    ExposedInPregnancy=("WasExposedInPregnancy", "any"),
    TotalExposureDaysInPregnancy=("PregnancyExposureDays", "sum"),
).reset_index()

pregnancy_df.head()

## 7. Assemble

In [ ]:
summary_df = (summary
              .merge(accumulated_delivery, on="PersonId")
              .merge(accumulated_preg_persistence, on="PersonId")
              .merge(pregnancy_df, on="PersonId"))

print("component sizes:", len(summary), len(accumulated_preg_persistence), len(pregnancy_df))
print("with zero pre-pregnancy persistence:",
      len(summary_df[summary_df["AccumulatedPersistenceBeforePregnancy"] == 0]))

summary_df["AccumulatedPersistenceBeforePregnancy"] = (
    summary_df["AccumulatedPersistenceBeforePregnancy"].fillna(0))
summary_df.head()

## 8. Descriptive tables

In [ ]:
!pip install tableone
from tableone import TableOne

In [ ]:
PERSISTENCE_COLUMNS = [
    "NumDrugEpisodes", "TotalDiscontinuationTime", "Reinitiation",
    "AccumulatedPersistenceBeforeDelivery", "AccumulatedPersistenceBeforePregnancy",
    "ExposedInPregnancy", "TotalExposureDaysInPregnancy",
]
PERSISTENCE_CATEGORICAL = ["Reinitiation", "ExposedInPregnancy"]

# Overall (no grouping).
TableOne(summary_df,
         columns=PERSISTENCE_COLUMNS + ["Discontinued60"],
         categorical=PERSISTENCE_CATEGORICAL + ["Discontinued60"],
         pval=False)

In [ ]:
# Grouped by whether the person ever discontinued for more than GAP_DAYS.
table1 = TableOne(summary_df, columns=PERSISTENCE_COLUMNS,
                  categorical=PERSISTENCE_CATEGORICAL,
                  groupby="Discontinued60", pval=True)
table1.to_csv(output_path_local + OUT_TABLE, index=True)
table1

In [ ]:
# Median [Q1, Q3] for the two headline persistence measures.
cols = ["AccumulatedPersistenceBeforePregnancy", "TotalExposureDaysInPregnancy"]
iqr_summary = summary_df[cols].agg(
    ["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
iqr_summary.index = ["Median", "Q1", "Q3"]
iqr_summary